In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
import warnings
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.multitest import multipletests
from scipy import stats
import matplotlib.patches as mpatches
from scipy.spatial.distance import cdist
from neuroCombat import neuroCombat

%matplotlib inline
warnings.filterwarnings('ignore')

In [2]:
# Define file paths for data, results, and output figures
data_path = os.path.dirname(os.getcwd()) + '/Data'
figure_path = os.path.dirname(os.getcwd()) + '/Figures'

In [3]:
# Load the data
df = pd.read_csv(data_path + '/validation/finland_validation_raw.csv', delimiter=',', index_col=0)
df = df[['SampleID', 'UniProt', 'Assay', 'NPX', 'Diagnosis', 'Subtype', 'Tissue', 'Batch']]

In [6]:
len(df.Assay.unique())

5440

## Normalise

This function removes batch effects from Olink data using ComBat. It first reshapes the data into a samples × proteins matrix, applies ComBat to adjust for batch-related biases, and then reshapes it back. The result is the same data structure, but all NPX values are normalized across batches so we can fairly compare samples measured in different experimental runs.

In [ ]:
duplicates = df[df.duplicated(subset=['SampleID', 'Assay'], keep=False)]

if not duplicates.empty:
    print(f"Found {len(duplicates)} duplicate rows for (SampleID, Assay):")
    print(duplicates.sort_values(['SampleID', 'Assay']).head(10))
else:
    print("No duplicate (SampleID, Assay) pairs found.")

In [ ]:
def batch_normalize(df):
    # Pivot to matrix: samples x proteins
    mat = df.pivot(index='SampleID', columns='Assay', values='NPX')
    
    # Prepare batch info
    batch_info = df[['SampleID', 'Batch']].drop_duplicates().set_index('SampleID')
    batch_info = batch_info.loc[mat.index]  # Ensure order matches
    
    # Apply ComBat (expects proteins x samples)
    combat_output = neuroCombat(
        dat=mat.T,
        covars=batch_info,
        batch_col='Batch'
    )
    
    # Get corrected matrix (back to samples x proteins)
    mat_corrected = pd.DataFrame(
        combat_output['data'].T,
        index=mat.index,
        columns=mat.columns
    )
    
    # Convert back to long format
    df_norm = mat_corrected.reset_index().melt(
        id_vars='SampleID',
        var_name='Assay',
        value_name='NPX'
    )
    
    # Add batch info back
    df_norm = df_norm.merge(
        df[['SampleID', 'Batch']].drop_duplicates(),
        on='SampleID'
    )
    
    return df_norm

df_norm = batch_normalize(df)

In [ ]:
# Merge with the original data
df_norm = df_norm.rename(columns={'NPX': 'NPX_norm'})

df_merged = df.merge(
    df_norm[['SampleID', 'Assay', 'NPX_norm']], 
    on=['SampleID', 'Assay'],                   
    how='left'                                 
)

In [ ]:
def plot_pca_comparison(df, color_col='Batch', title_color='Batch'):

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle(f"PCA Comparison: Before vs After Normalization ({title_color})", fontsize=14)

    for i, (value_col, ax, title) in enumerate([
        ('NPX', axes[0], 'Before Normalization'),
        ('NPX_norm', axes[1], 'After Normalization')
    ]):
        # Pivot to samples × proteins
        pivot_data = df.pivot(index='SampleID', columns='Assay', values=value_col)

        # Standardize features
        scaled_data = StandardScaler().fit_transform(pivot_data)

        # PCA
        pca = PCA(n_components=2)
        coords = pca.fit_transform(scaled_data)

        # Attach metadata
        sample_meta = df.groupby('SampleID')[[color_col]].first().loc[pivot_data.index]

        # Plot each group (Batch or Diagnosis/subtype)
        for group in sorted(sample_meta[color_col].dropna().unique()):
            mask = sample_meta[color_col] == group
            ax.scatter(
                coords[mask, 0],
                coords[mask, 1],
                label=f'{color_col}: {group}',
                alpha=0.7,
                s=30
            )

        ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
        ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
        ax.set_title(title)
        ax.grid(alpha=0.3)
        ax.legend()

    plt.tight_layout()
    plt.show()

plot_pca_comparison(df_merged, color_col='Batch', title_color='Batch')
plot_pca_comparison(df_merged, color_col='Diagnosis', title_color='Diagnosis')
plot_pca_comparison(df_merged, color_col='Subtype', title_color='Subtype')

# Calculate Effect sizes

In [ ]:
# # Filter based on selected 19 proteins
# discovery_data = pd.read_excel(data_path + '/results/HeH_vs_ETV6_RUNX1_results.xlsx')  # Original discovery
# protein_list = list(discovery_data['Protein/gene'])

In [ ]:
# If this results file is not available define the proteins directly
protein_list = ['DSC2','PTPRK', 'IL6R','RGMA','CCL17', 'BTN1A1','ADAM8', 'S100A16', 'CEACAM6', 
                'LAT2','AKAP12','LSP1','IL7R','FLT3LG','IL3','LSM8','CRH','BLNK','NOS2']

In [ ]:
# Select B-ALL and our proteins
# Alternatively we could run this for all proteins but it takes long time to run
df_merged = df_merged[df_merged['Diagnosis'] == 'pre-B-ALL']
df_merged = df_merged[df_merged['Assay'].isin(protein_list)]

# Rename the subtype
df_merged['Subtype'] = df_merged['Subtype'].replace('ER', 'ETV6::RUNX1')

# Separate based on tissue
pb = df_merged[df_merged['Tissue'] == 'peripheral_blood']
bm = df_merged[df_merged['Tissue'] == 'bone_marrow']

pb.to_csv('../data/results/validation/pb_olink.csv', index=False)
bm.to_csv('../data/results/validation/bm_olink.csv', index=False)

In [ ]:
print("\nPeripheral Blood - Patients by Subtype:")
print(pb.groupby('Subtype')['SampleID'].nunique())

print("\nBone Marrow - Patients by Subtype:")
print(bm.groupby('Subtype')['SampleID'].nunique())

In [ ]:
def calculate_stats(df, npx_col='NPX'):
    df_filtered = df[df['Subtype'].isin(['HeH', 'ETV6::RUNX1'])].copy()
    
    results = []
    
    for protein in df_filtered['Assay'].unique():
        protein_data = df_filtered[df_filtered['Assay'] == protein]
        
        # Get values for each subtype
        heh_values = protein_data[protein_data['Subtype'] == 'HeH'][npx_col].dropna()
        etv6_values = protein_data[protein_data['Subtype'] == 'ETV6::RUNX1'][npx_col].dropna()
        
        # Calculate averages
        avg_heh = heh_values.mean() if len(heh_values) > 0 else np.nan
        avg_etv6 = etv6_values.mean() if len(etv6_values) > 0 else np.nan
        
        # Log2FC (HeH - ETV6::RUNX1)
        log2fc = avg_heh - avg_etv6 if not np.isnan(avg_heh) and not np.isnan(avg_etv6) else np.nan
        
        # Cohen's d
        if len(heh_values) > 0 and len(etv6_values) > 0:
            pooled_std = np.sqrt(((len(heh_values)-1)*heh_values.std()**2 + 
                                 (len(etv6_values)-1)*etv6_values.std()**2) / 
                                (len(heh_values) + len(etv6_values) - 2))
            cohen_d = abs(log2fc) / pooled_std if pooled_std > 0 else np.nan
        else:
            cohen_d = np.nan
        
        # P-value (t-test)
        if len(heh_values) > 1 and len(etv6_values) > 1:
            _, p_value = stats.mannwhitneyu(heh_values, etv6_values, alternative='two-sided')
        else:
            p_value = np.nan
        
        results.append({
            'Protein/gene': protein,
            'Average HeH': avg_heh,
            'Average ETV6::RUNX1': avg_etv6,
            'Log2FC': log2fc,
            'Effect size': cohen_d,
            'P-value': p_value
        })
    
    # Create results dataframe
    results_df = pd.DataFrame(results)
    
    # Adjusted p-value (FDR correction)
    if results_df['P-value'].notna().sum() > 0:
        pvals = results_df['P-value'].fillna(1)
        _, padj, _, _ = multipletests(pvals, method='fdr_bh')
        results_df['Adjusted P-value'] = padj
        results_df.loc[results_df['P-value'].isna(), 'Adjusted P-value'] = np.nan
    else:
        results_df['Adjusted P-value'] = np.nan
    
    return results_df

# Calculate for each tissue and NPX type
pb_npx = calculate_stats(pb, 'NPX')
pb_npx_norm = calculate_stats(pb, 'NPX_norm')
bm_npx = calculate_stats(bm, 'NPX')
bm_npx_norm = calculate_stats(bm, 'NPX_norm')

# Save results
pb_npx.to_csv('../data/results/validation/pb_npx_results.csv', index=False)
pb_npx_norm.to_csv('../data/results/validation/pb_npx_norm_results.csv', index=False)
bm_npx.to_csv('../data/results/validation/bm_npx_results.csv', index=False)
bm_npx_norm.to_csv('../data/results/validation/bm_npx_norm_results.csv', index=False)

In [ ]:
# Add dataset labels
pb_npx['Dataset'] = 'PB NPX'
pb_npx_norm['Dataset'] = 'PB NPX_norm'
bm_npx['Dataset'] = 'BM NPX'
bm_npx_norm['Dataset'] = 'BM NPX_norm'

# Create significance categories
def get_significance_stars(p_val):
    if pd.isna(p_val):
        return 'ns'
    elif p_val < 0.001:
        return '***'
    elif p_val < 0.01:
        return '**'
    elif p_val < 0.05:
        return '*'
    else:
        return 'ns'

# Add significance stars
for df in [pb_npx, pb_npx_norm, bm_npx, bm_npx_norm]:
    df['Significance'] = df['Adjusted P-value'].apply(get_significance_stars)

# Color coding function
def get_color(log2fc, adj_pval):
    if pd.isna(adj_pval) or adj_pval >= 0.05:
        return '#D3D3D3'    # Light gray for not significant
    elif log2fc > 0:
        return '#FF8C42'    # Orange for HeH 
    else:
        return '#6A4C93'    # Purple for ETV6::RUNX1 

# Add colors
for df in [pb_npx, pb_npx_norm, bm_npx, bm_npx_norm]:
    df['Color'] = df.apply(lambda row: get_color(row['Log2FC'], row['Adjusted P-value']), axis=1)

# Get proteins that are present in all datasets
common_proteins = set(pb_npx['Protein/gene'])
for df in [pb_npx_norm, bm_npx, bm_npx_norm]:
    common_proteins = common_proteins.intersection(set(df['Protein/gene']))

In [ ]:
# OPTION 1: Show all proteins
show_all = True  # Change to False to show only top proteins

# OPTION 2: Filter to top N most significant proteins (by minimum p-value across datasets)
if not show_all:
    # Combine p-values and get minimum for each protein
    all_pvals = {}
    for protein in common_proteins:
        pvals = []
        for df in [pb_npx, pb_npx_norm, bm_npx, bm_npx_norm]:
            p = df[df['Protein/gene'] == protein]['Adjusted P-value'].values
            if len(p) > 0:
                pvals.append(p[0])
        all_pvals[protein] = np.nanmin(pvals) if pvals else 1.0
    
    # Select top 19 most significant proteins
    top_n = 19
    sorted_proteins = sorted(all_pvals.items(), key=lambda x: x[1])
    common_proteins = set([p[0] for p in sorted_proteins[:top_n]])
    print(f"Showing top {top_n} most significant proteins")

# Filter to common proteins and sort by PB NPX effect size
pb_npx_filtered = pb_npx[pb_npx['Protein/gene'].isin(common_proteins)].copy()
protein_order = pb_npx_filtered.sort_values('Effect size', ascending=True)['Protein/gene'].values

# Adjust figure height based on number of proteins
fig_height = max(8, len(protein_order) * 0.3)
fig, axes = plt.subplots(1, 4, figsize=(20, fig_height), sharey=True)

y_pos = np.arange(len(protein_order))

# Function to plot data
def plot_forest(ax, data, protein_order, title, show_ylabel=False):
    # Reindex data by protein order
    data_indexed = data.set_index('Protein/gene').loc[protein_order]
    
    # Plot bars
    bars = ax.barh(y_pos, data_indexed['Log2FC'], height=0.6,
                   color=data_indexed['Color'], alpha=0.8,
                   edgecolor='white', linewidth=1)
    
    # Add significance stars and effect sizes
    for i, (protein, row) in enumerate(data_indexed.iterrows()):
        # Significance stars
        if row['Significance'] != 'ns':
            x_pos = row['Log2FC'] + (0.1 if row['Log2FC'] > 0 else -0.1)
            ax.text(x_pos, i, row['Significance'],
                   ha='left' if row['Log2FC'] > 0 else 'right',
                   va='center', fontsize=9)
        
        # Effect size text (only for larger effects)
        if abs(row['Log2FC']) > 0.5:
            ax.text(row['Log2FC']/2, i, f'{row["Effect size"]:.1f}',
                   ha='center', va='center',
                   fontsize=8, color='white', weight='bold')
    
    # Customize plot
    ax.axvline(x=0, color='gray', linestyle='-', linewidth=1)
    ax.set_xlabel('Log2FC', fontsize=11)
    ax.set_title(title, fontsize=12, pad=10, weight='bold')
    ax.grid(True, alpha=0.3, axis='x')
    
    # Set y-axis labels only for leftmost plot
    if show_ylabel:
        ax.set_yticks(y_pos)
        ax.set_yticklabels(protein_order, fontsize=9)
    
    # Set x-axis limits for consistency
    max_abs = max(abs(data_indexed['Log2FC'].min()), abs(data_indexed['Log2FC'].max()))
    ax.set_xlim(-max_abs*1.2, max_abs*1.2)

# Plot all four datasets
plot_forest(axes[0], pb_npx[pb_npx['Protein/gene'].isin(common_proteins)], 
            protein_order, 'Peripheral Blood\n(NPX)', show_ylabel=True)
plot_forest(axes[1], pb_npx_norm[pb_npx_norm['Protein/gene'].isin(common_proteins)], 
            protein_order, 'Peripheral Blood\n(NPX normalized)', show_ylabel=False)
plot_forest(axes[2], bm_npx[bm_npx['Protein/gene'].isin(common_proteins)], 
            protein_order, 'Bone Marrow\n(NPX)', show_ylabel=False)
plot_forest(axes[3], bm_npx_norm[bm_npx_norm['Protein/gene'].isin(common_proteins)], 
            protein_order, 'Bone Marrow\n(NPX normalized)', show_ylabel=False)

# Overall title
fig.suptitle('', fontsize=14, y=1.01, weight='bold')

# Create legend
legend_elements = [
    mpatches.Patch(color='#FF8C42', alpha=0.8, label='Higher in HeH (adj.p<0.05)'),
    mpatches.Patch(color='#6A4C93', alpha=0.8, label='Higher in ETV6::RUNX1 (adj.p<0.05)'),
    mpatches.Patch(color='#D3D3D3', alpha=0.8, label='Not significant')
]

# Place legend
legend = axes[3].legend(handles=legend_elements,
                        loc='center left',
                        bbox_to_anchor=(0.1, -0.185),
                        fancybox=True,
                        shadow=True,
                        fontsize=10)

legend.get_frame().set_facecolor('white')
legend.get_frame().set_edgecolor('lightgray')
legend.get_frame().set_alpha(0.9)

# Add significance legend
significance_text = '*** adj.p<0.001, ** adj.p<0.01, * adj.p<0.05'
fig.text(0.784, -0.07, significance_text, ha='center', fontsize=10,
         bbox=dict(boxstyle='round,pad=0.5',
                  facecolor='#F8F8F8',
                  edgecolor='lightgray',
                  alpha=0.9))

plt.tight_layout()
plt.subplots_adjust(bottom=0.03, right=0.88)
plt.savefig('forest_plot_normalization_comparison.png', dpi=300, bbox_inches='tight')
plt.show()